In [1]:

# Importar librerías necesarias
import requests
from bs4 import BeautifulSoup
import time
import os
from pyspark.sql import SparkSession

# Inicializar SparkSession si aún no está inicializada
# En Databricks o Google Colab con pyspark, esto ya suele estar configurado.
try:
    spark = SparkSession.builder.appName("GutenbergDownloader").getOrCreate()
    print("SparkSession lista.")
except Exception as e:
    print(f"Error al obtener SparkSession: {e}")
    # Si falla, simplemente continuamos con el código Python/Scraping
    pass

# --- Configuración de Archivo y Carpetas ---

# Carpeta donde se guardarán los libros. 
# Si el notebook está en un entorno distribuido (Databricks, EMR), 
# esta carpeta se creará en el sistema de archivos local del nodo driver.
folder = "gutenberg_books"
os.makedirs(folder, exist_ok=True)
print(f"Carpeta de destino: {os.path.abspath(folder)}")

# URL de los libros más descargados
url = "https://www.gutenberg.org/browse/scores/top"

# --- Web Scraping y Extracción de Enlaces ---

print("\n1. Obteniendo la lista de los libros más descargados...")
try:
    response = requests.get(url, timeout=15)
    response.raise_for_status() # Lanza excepción para códigos de error HTTP
    soup = BeautifulSoup(response.text, "html.parser")

    # Obtener lista de libros (primer <ol> de la sección "Most Frequently Downloaded")
    # Utilizamos find(id=...) para ser más específicos
    content = soup.find(id="content")
    ol = content.find("ol") if content else soup.find("ol")
    
    if ol:
        book_links = ol.find_all("a")
        print(f"Encontrados {len(book_links)} posibles enlaces de libros.")
    else:
        print("Error: No se encontró la lista de libros (<ol>).")
        book_links = []

except requests.exceptions.RequestException as e:
    print(f"Error de solicitud HTTP: {e}")
    book_links = []
except Exception as e:
    print(f"Error general en scraping: {e}")
    book_links = []


# --- Descarga de Libros ---

print("\n2. Iniciando descarga de los primeros 100 libros...")
books_to_download = book_links[:100]

for i, book in enumerate(books_to_download, start=1):
    try:
        book_name = book.text.strip()
        # El ID es el último segmento de la URL del enlace del libro
        book_id = book['href'].split('/')[-1]

        # Limpiar nombre de archivo (reemplazamos caracteres no seguros)
        safe_name = "".join(c for c in book_name if c.isalnum() or c in " _-").rstrip()
        file_path = os.path.join(folder, f"{safe_name}.txt")

        # Saltar si ya existe
        if os.path.exists(file_path):
            print(f"[{i:03}] {book_name} ya existe, saltando...")
            continue

        # Posibles URLs de descarga del archivo .txt
        urls_to_try = [
            f"https://www.gutenberg.org/files/{book_id}/{book_id}-0.txt", # Típico para UTF-8
            f"https://www.gutenberg.org/files/{book_id}/{book_id}-8.txt", # Típico para Latin-1
            f"https://www.gutenberg.org/files/{book_id}/{book_id}-1.txt", # Otras codificaciones
            f"https://www.gutenberg.org/files/{book_id}/{book_id}.txt",   # Archivo simple
        ]

        downloaded = False
        for txt_url in urls_to_try:
            try:
                # Intentamos descargar el archivo .txt
                r = requests.get(txt_url, timeout=10)
                if r.status_code == 200:
                    # Guardamos el contenido binario (wb) para manejar cualquier codificación
                    with open(file_path, "wb") as f:
                        f.write(r.content)
                    print(f"[{i:03}] Descargado: {book_name}")
                    downloaded = True
                    time.sleep(1)  # PAUSA CORTA: Requisito de cortesía para no saturar el servidor de Gutenberg
                    break # Salir del bucle de URLs a intentar
            except requests.exceptions.RequestException as req_err:
                # Error de solicitud o timeout al intentar esta URL específica
                # print(f"[{i:03}] Error al intentar {txt_url}: {req_err}") # Comentado para no saturar el log
                pass
            except Exception as e:
                 print(f"[{i:03}] Error inesperado con {txt_url}: {e}")
                 pass

        if not downloaded:
            print(f"[{i:03}] No se encontró un archivo .txt descargable para: {book_name}")

    except Exception as e:
        print(f"[{i:03}] Fallo general al procesar el libro {book.text.strip()}: {e}")


print("\n--- Proceso de descarga completado! ---")
print("\n++++ danielmedina +++++")
# Opcional: Mostrar la lista de archivos descargados
print(f"Archivos guardados en: {os.path.abspath(folder)}")
# print(os.listdir(folder)) # Descomentar para ver la lista de archivos

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/13 23:09:59 WARN Utils: Your hostname, DMPC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/02/13 23:09:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/13 23:10:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession lista.
Carpeta de destino: /home/daniel/proyecto_abundis/proyecto_abundis/gutenberg_books

1. Obteniendo la lista de los libros más descargados...
Encontrados 100 posibles enlaces de libros.

2. Iniciando descarga de los primeros 100 libros...
[001] Frankenstein; Or, The Modern Prometheus by Mary Wollstonecraft Shelley (5913) ya existe, saltando...
[002] Wuthering Heights by Emily Brontë (4109) ya existe, saltando...
[003] Moby Dick; Or, The Whale by Herman Melville (4097) ya existe, saltando...
[004] Pride and Prejudice by Jane Austen (3121) ya existe, saltando...
[005] Romeo and Juliet by William Shakespeare (2705) ya existe, saltando...
[006] The Strange Case of Dr. Jekyll and Mr. Hyde by Robert Louis Stevenson (2236) ya existe, saltando...
[007] A Room with a View by E. M.  Forster (2169) ya existe, saltando...
[008] The Complete Works of William Shakespeare by William Shakespeare (2163) ya existe, saltando...
[009] Alice's Adventures in Wonderland by Lewis Carroll (20

In [2]:
#---------Vectores mediante TF-IDF-----------

# CELDA 1: Obtencion de datos

from pyspark.sql import SparkSession
from pyspark.ml.feature import Tokenizer, CountVectorizer, IDF
from pyspark.sql.functions import input_file_name, regexp_extract, udf, col # <--- AGREGAR udf y col
from pyspark.sql.types import StringType # <--- AGREGAR StringType
from urllib.parse import unquote # <--- IMPORTAR el decodificador de Python

# 1. Obtener la SparkSession (ya debe estar activa)
spark = SparkSession.builder.getOrCreate()
data_path = "gutenberg_books/*.txt"

# --- 2. Cargar y Preparar los Documentos ---

# Leer cada archivo como un documento completo 
documents_df = spark.read.text(data_path, wholetext=True)

# Muestra la ruta SÓLO si se quiere verificar 
#input_file_name() 
#documents_df.select(input_file_name()).show(2, truncate=False)

# --- Función UDF para Decodificar URL ---
def url_decode_name(name):
    """Decodifica el nombre de archivo de formato URL (%20, etc.)."""
    if name is not None:
        return unquote(name)
    return name

# Registrar la función como UDF de PySpark
decode_udf = udf(url_decode_name, StringType())
# ----------------------------------------

# Lógica de extracción de doc_id con la corrección:
# El patrón r'[^/\\?]+$' extrae cualquier cosa después del último '/' o '\'

# 1. Extraer el nombre de archivo codificado
documents_df = documents_df.withColumn(
    "doc_id", 
    regexp_extract(input_file_name(), r'[^/\\?]+$', 0)
).withColumnRenamed("value", "text")

# 2. Aplicar la decodificación para limpiar el nombre
documents_df = documents_df.withColumn(
    "doc_id", 
    decode_udf(col("doc_id")) # Llama al UDF sobre la columna
)

print("Documentos cargados con IDs extraídos.")
# Verifica que esta línea te muestre los nombres correctos:
documents_df.select("doc_id").show(5, truncate=False)




ModuleNotFoundError: No module named 'numpy'

In [ ]:
#---------Vectores mediante TF-IDF-----------

# CELDA 2: PREPROCESAMIENTO (NLTK EN DRIVER)

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import pandas as pd

# Descargar SOLO UNA VEZ
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

stop_words_nltk = set(stopwords.words("english"))

# Pasar SOLO lo necesario a Pandas
pdf = documents_df.select("doc_id", "text").toPandas()

print("Documentos pasados a Pandas para preprocesamiento.")

# --- Función de limpieza ---
def preprocess_text_nltk(text):
    if text is None:
        return ""
    tokens = word_tokenize(text.lower())
    tokens = [
        w for w in tokens
        if w.isalnum() and w not in stop_words_nltk and len(w) > 2
    ]
    return " ".join(tokens)

# Aplicar limpieza
pdf["clean_text"] = pdf["text"].apply(preprocess_text_nltk)

# Regresar a Spark SOLO con texto limpio
documents_df = spark.createDataFrame(
    pdf[["doc_id", "clean_text"]]
)

print("Preprocesamiento aplicado correctamente.")


In [ ]:
#---------Vectores mediante TF-IDF-----------

# CELDA 3: TF-IDF MANUAL CON PYSPARK

from pyspark.sql.functions import (
    col, lit, split, explode,
    length, log, countDistinct
)

# Tokenización desde texto limpio
df_tokens = documents_df.withColumn(
    "word",
    explode(split(col("clean_text"), "\\s+"))
).filter(length(col("word")) > 2)

print("Tokenización completada.")

# TF
tf_df = df_tokens.groupBy("doc_id", "word").count() \
    .withColumnRenamed("count", "tf")

print("TF calculado.")

# Número total de documentos
N = documents_df.select("doc_id").distinct().count()
print(f"Número total de documentos: {N}")

# DF
df_df = df_tokens.groupBy("word") \
    .agg(countDistinct("doc_id").alias("df"))

print("DF calculado.")

# IDF
idf_df = df_df.withColumn(
    "idf",
    log(lit(1) + (lit(N) / col("df")))
)

print("IDF calculado.")

# TF-IDF
tfidf_df = tf_df.join(idf_df, "word") \
    .withColumn("tfidf", col("tf") * col("idf"))

print("TF-IDF calculado correctamente.")


In [ ]:
# --------- MATRIZ DE SIMILITUD DEL COSENO (MANUAL, SOLO SPARK) ---------

from pyspark.sql.functions import col, sqrt, sum as spark_sum

print("--- INICIANDO CÁLCULO MANUAL DE SIMILITUD DEL COSENO ---")

# tfidf_df debe tener: doc_id | word | tfidf

# 1. Calcular la norma de cada documento: ||d||
norms_df = tfidf_df.groupBy("doc_id").agg(
    sqrt(spark_sum(col("tfidf") * col("tfidf"))).alias("norm")
)

print("Normas de documentos calculadas.")

# 2. Producto punto entre pares de documentos
dot_product_df = (
    tfidf_df.alias("a")
    .join(
        tfidf_df.alias("b"),
        on="word"
    )
    .select(
        col("a.doc_id").alias("doc_i"),
        col("b.doc_id").alias("doc_j"),
        (col("a.tfidf") * col("b.tfidf")).alias("prod")
    )
    .groupBy("doc_i", "doc_j")
    .agg(
        spark_sum("prod").alias("dot_product")
    )
)

print("Producto punto entre documentos calculado.")

# 3. Unir normas para cada par
similarity_df = (
    dot_product_df
    .join(norms_df.withColumnRenamed("doc_id", "doc_i"), on="doc_i")
    .withColumnRenamed("norm", "norm_i")
    .join(norms_df.withColumnRenamed("doc_id", "doc_j"), on="doc_j")
    .withColumnRenamed("norm", "norm_j")
    .withColumn(
        "cosine_similarity",
        col("dot_product") / (col("norm_i") * col("norm_j"))
    )
)

print("Similitud del coseno calculada manualmente.")

# 4. Mostrar ejemplo
similarity_df.select(
    "doc_i", "doc_j", "cosine_similarity"
).orderBy(col("cosine_similarity").desc()).show(10, truncate=False)


In [ ]:
#---------Listado de libros disponibles-----------

from pyspark.sql.functions import col, regexp_replace

print("\n=== LIBROS DISPONIBLES (COPIA EL NOMBRE EXACTO) ===\n")

(
    documents_df
    .select("doc_id")
    .distinct()
    .orderBy("doc_id")
    .withColumn(
        "Libro (copiar tal cual)",
        regexp_replace(col("doc_id"), r"\s+$", "")
    )
    .select("Libro (copiar tal cual)")
    .show(50, truncate=False)  # muestra 50, puedes ajustar
)

print("\nSi no aparece el libro que buscas, aumenta el número mostrado.\n")


In [ ]:
#---------Recomendacion de 10 libros-----------

# CELDA 1: DEFINICIÓN DE LA FUNCIÓN

from pyspark.sql.functions import col, desc

def recomendar_libros_spark(doc_referencia, similitud_df, n=10):
    """
    Recomienda n libros usando Spark
    """

    recomendaciones = (
        similitud_df
        .filter(col("doc_i") == doc_referencia)
        .filter(col("doc_j") != doc_referencia)
        .orderBy(desc("cosine_similarity"))
        .limit(n)
    )

    return recomendaciones


In [ ]:
#---------Recomendacion de 10 libros-----------

# CELDA 2: INTERFAZ INTERACTIVA Y EJECUCIÓN (CORREGIDA)

from pyspark.sql.functions import col, regexp_replace

# --- Mostrar ejemplos (limitado para no saturar memoria) ---
print("Ejemplos de libros disponibles:")
similarity_df.select("doc_i").distinct().limit(10).show(truncate=False)

# --- Entrada del usuario (driver) ---
doc_elegido = input("Ingresa el nombre EXACTO del libro de referencia: ")

# --- Obtener MÁS candidatos para evitar perder libros al limpiar ---
top_n = recomendar_libros_spark(
    doc_referencia=doc_elegido,
    similitud_df=similarity_df,
    n=30  # pedimos más para luego limpiar
)

# --- Eliminar el mismo libro (no es recomendación) ---
top_n = top_n.filter(col("doc_j") != doc_elegido)

# --- Normalizar título para detectar duplicados ---
top_10_limpio = (
    top_n
    .withColumn(
        "titulo_base",
        regexp_replace(col("doc_j"), r"\s\d+\.txt$", "")
    )
    .dropDuplicates(["titulo_base"])
    .limit(10)
)

# --- Mostrar resultado final ---
print("\n--- TOP 10 LIBROS RECOMENDADOS ---")
top_10_limpio.select(
    col("titulo_base").alias("Libro recomendado"),
    col("cosine_similarity").alias("Similitud")
).show(truncate=False)



In [ ]:
#---------Resumen en 20 palabras-----------

# CELDA 1: DEFINICIÓN DE LA FUNCIÓN

from pyspark.sql.functions import col, desc

def resumen_20_palabras_spark(doc_id, tfidf_df, n=20):
    """
    Genera un resumen usando las 20 palabras con mayor TF-IDF
    """

    resumen_df = (
        tfidf_df
        .filter(col("doc_id") == doc_id)
        .orderBy(desc("tfidf"))
        .limit(n)
        .select("word")
    )

    palabras = [row.word for row in resumen_df.collect()]
    resumen = " ".join(palabras)

    return resumen



In [ ]:
#---------Resumen en 20 palabras-----------

# CELDA 2: INTERFAZ INTERACTIVA PARA EL RESUMEN

#---------Resumen en 20 palabras-----------

from pyspark.sql.functions import col

# Mostrar ejemplos de libros disponibles
print("Ejemplos de libros disponibles (copia EXACTAMENTE uno):")
print("-------------------------------------------------------")

tfidf_df.select("doc_id").distinct().show(10, truncate=False)

# Entrada del usuario
doc_elegido = input("Ingresa el nombre EXACTO del libro: ")

print(f"\n--- RESUMEN (20 palabras con mayor TF-IDF) ---")

# Validación
if tfidf_df.filter(col("doc_id") == doc_elegido).count() == 0:
    print("Error: El libro no existe en el catálogo.")
else:
    resumen = resumen_20_palabras_spark(
        doc_id=doc_elegido,
        tfidf_df=tfidf_df,
        n=20
    )

    print(resumen)
    print(f"\n(Palabras: {len(resumen.split())})")
